# UNet++ RD classifier: GPU resource check

This is the next notebook after `baseline_training.ipynb`. Its purpose is to
measure a **paper-relevant model at the paper's full input resolution** on your
available Colab GPU. It does not train the 50-epoch benchmark yet, and it does
not run any test evaluation. The previous custom 16-frame pilot is preserved as
engineering evidence, not relabeled as a paper reproduction.

Connect this notebook to a Colab CUDA GPU (T4 if available) and run from the top.
It is standalone and does not need the previous notebook's Python variables.
Only two official-training clips (about 0.5 MB) are needed. The authored model
uses 96 frames and 3,603,553 parameters, compared with the pilot's 16 frames and
17,585 parameters. Its five encoder blocks and classifier follow the pinned
release; it is not a generic nested segmentation UNet++.

**Protocol conflict:** the paper states AdamW, while the released YAML uses Adam.
This configuration explicitly follows the released code. It also measures
physical batches before proposing accumulation to retain the effective batch 48.
Those adaptations must be disclosed. See the
[source audit](../docs/research/erdes-unetpp-protocol.md).

All benchmark weights are discarded. A successful memory test or a timing
projection is not a completed reproduction or a model-performance result.


## 1. GPU and persistent result storage

Google Drive persistence is enabled below for Colab. Mount/sign in when prompted.
Results will be copied directly to **My Drive → rd-ultrasound-ml → outputs →
unetpp_resource_check**. Raw clips remain in runtime storage. Connecting
Drive alone does not move existing baseline files; preserve their exported ZIPs.
If you intentionally disable Drive, download the final ZIP before disconnecting.


In [2]:
import gc
import hashlib
import io
import json
import random
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import FileLink, display

if not torch.cuda.is_available():
    raise RuntimeError('Connect a Colab CUDA GPU, then run this notebook from the top.')
UNETPP_USE_GOOGLE_DRIVE = True
unetpp_cwd = Path.cwd()
unetpp_project_root = next((p for p in (unetpp_cwd,*unetpp_cwd.parents)
    if (p/'pyproject.toml').is_file() and (p/'notebooks').is_dir()), None)
unetpp_is_colab = Path('/content').is_dir()
if unetpp_project_root is None:
    if not unetpp_is_colab:
        raise RuntimeError('Open from the project folder or use a Colab runtime.')
    unetpp_project_root = Path('/content')
unetpp_data_dir = (Path('/content') if unetpp_is_colab else unetpp_project_root)/'data/erdes'
unetpp_results_root = (Path('/content') if unetpp_is_colab else unetpp_project_root)/'outputs/unetpp_resource_check'
if unetpp_is_colab and UNETPP_USE_GOOGLE_DRIVE:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Drive did not mount; stop before creating result folders.')
    unetpp_results_root = Path('/content/drive/MyDrive/rd-ultrasound-ml/outputs/unetpp_resource_check')
unetpp_run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
UNETPP_OUTPUT_DIR = unetpp_results_root/unetpp_run_id
UNETPP_OUTPUT_DIR.mkdir(parents=True,exist_ok=False)
unetpp_data_dir.mkdir(parents=True,exist_ok=True)

def unetpp_save_json(name,value):
    temporary=UNETPP_OUTPUT_DIR/(name+'.tmp')
    temporary.write_text(json.dumps(value,indent=2,allow_nan=False)+'\n',encoding='utf-8')
    temporary.replace(UNETPP_OUTPUT_DIR/name)

unetpp_runtime={'python':sys.version,'torch':str(torch.__version__),'numpy':np.__version__,
    'pandas':pd.__version__,'opencv':cv2.__version__,'cuda':torch.version.cuda,
    'cudnn':torch.backends.cudnn.version(),'gpu':torch.cuda.get_device_name(0),
    'gpu_total_bytes':torch.cuda.get_device_properties(0).total_memory,
    'cpu_torch_threads':torch.get_num_threads()}
print(unetpp_runtime)
print('Results:',UNETPP_OUTPUT_DIR)


Mounted at /content/drive
{'python': '3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'pandas': '2.2.3', 'opencv': '4.14.0', 'cuda': '13.0', 'cudnn': 92700, 'gpu': 'Tesla T4', 'gpu_total_bytes': 15637086208, 'cpu_torch_threads': 1}
Results: /content/drive/MyDrive/rd-ultrasound-ml/outputs/unetpp_resource_check/20261101T193236373065Z


## 2. Record the intended full-data protocol

The embedded proposal matches `configs/unetpp_rd_001.json`. The final research
training target is all 3,873 official training clips, all 431 validation clips,
and 50 epochs. This preflight does not acquire that full data or claim to complete
it. The paper's test numbers are reference values only; do not compare our pilot
validation accuracy with the paper's test accuracy as if they were equivalent.


In [3]:
UNETPP_CONFIG = {'experiment_id': 'unetpp_rd_001',
 'status': 'resource_check_only_not_trained',
 'scope': 'one published UNet++ non-RD vs RD baseline; not all 40 benchmark runs',
 'reference_article': 'https://arxiv.org/html/2508.04735v2',
 'upstream_revision': '4af67e0b9f4667edbae75a64a819884ee14d7a3b',
 'reference_experiment': 'https://github.com/OSUPCVLab/ERDES/blob/4af67e0b9f4667edbae75a64a819884ee14d7a3b/configs/experiment/non_rd_vs_rd/unetplusplus.yaml',
 'dataset_revision': '7867e2504ed3845a46ff7790a2f8c3ffcdc08947',
 'task': 'non_rd_vs_rd',
 'label_mapping': {'non_rd': 0, 'rd': 1},
 'seed': 42,
 'model': {'name': 'released_UNetPlusPlus_encoder_classifier_equivalent',
           'features': [32, 32, 64, 128, 256],
           'convolutions_per_block': 2,
           'conv': {'kernel_size': 3, 'stride': 1, 'padding': 1, 'bias': True},
           'downsampling': 'four MaxPool3d(kernel_size=2,stride=2)',
           'normalization': 'InstanceNorm3d(eps=1e-5,momentum=0.1,affine=True,track_running_stats=False)',
           'activation': 'LeakyReLU(negative_slope=0.1,inplace=True)',
           'dropout': 0.0,
           'head': 'AdaptiveAvgPool3d(1), Linear(256,128), ReLU, Linear(128,1)',
           'expected_parameter_count': 3603553,
           'pretrained': False},
 'preprocessing': {'num_frames': 96,
                   'height': 128,
                   'width': 128,
                   'grayscale': 'arithmetic channel mean',
                   'padding': 'symmetric zero padding to square',
                   'resize': 'full time/spatial trilinear interpolation',
                   'align_corners': False,
                   'normalization': 'divide by 255',
                   'dtype': 'float32',
                   'layout': 'B,C,D,H,W',
                   'frame_subsampling': False,
                   'augmentation': 'none',
                   'roundoff_tolerance': 1e-06},
 'intended_training': {'train_clips': 3873,
                       'validation_clips': 431,
                       'epochs': 50,
                       'optimizer': 'Adam',
                       'learning_rate': 1.5e-05,
                       'betas': [0.9, 0.999],
                       'eps': 1e-08,
                       'weight_decay': 0.0,
                       'scheduler': 'ReduceLROnPlateau(mode=min,factor=0.1,patience=10)',
                       'scheduler_monitor': 'validation_loss',
                       'loss': 'unweighted BCEWithLogitsLoss',
                       'class_weighting': False,
                       'resampling': False,
                       'reference_effective_batch': 48,
                       'physical_batch': 'pending GPU preflight',
                       'gradient_accumulation': '48 / physical batch, subject to '
                                                'measured fit',
                       'precision': 'float32',
                       'deterministic_algorithms': False,
                       'checkpoint_selection': 'highest validation accuracy; earliest '
                                               'on ties'},
 'evaluation': {'positive_class': 'rd',
                'threshold': 0.5,
                'reference_test_clips': 1077,
                'unetpp_metrics': ['sensitivity',
                                   'specificity',
                                   'precision',
                                   'F1',
                                   'accuracy'],
                'test_policy': 'manual evaluation after protocol/checkpoint freeze; no '
                               'automatic test in preflight'},
 'published_reference': {'table': '8, Non-RD vs RD columns, UNet++ row',
                         'partition': 'test',
                         'sensitivity': 0.879,
                         'specificity': 0.986,
                         'precision': 0.871,
                         'F1': 0.875,
                         'accuracy': 0.976,
                         'source': 'author-reported, not our result'},
 'disclosures': ['Paper says AdamW; released experiment resolves to Adam. This first '
                 'attempt follows pinned code, not a silently reconciled paper '
                 'protocol.',
                 'One-GPU gradient accumulation and current dependency versions must '
                 'be reported as hardware/software adaptations.',
                 'Independent PyTorch blocks replace MONAI wrappers; verify parameter '
                 'and numerical parity against the read-only reference.',
                 'OpenCV decoding replaces torchvision/PyAV; grayscale/interpolation '
                 'parity is checked, but cross-version decoder identity is not '
                 'guaranteed.',
                 'Only float32 roundoff within 1e-6 is clipped at normalization '
                 'boundaries.',
                 'Paper recall decimals do not reconcile with a simple 100-positive '
                 'test denominator; exact reported-table agreement is not guaranteed.',
                 'Official splits are clip-level with possible same-patient overlap; '
                 'no unseen-patient or clinical-validation claim.'],
 'preflight': {'physical_batches': [1, 2, 4],
               'warmup_steps': 1,
               'measured_steps': 3,
               'source_clips': 2,
               'full_training_performed': False,
               'test_used': False}}

random.seed(UNETPP_CONFIG['seed'])
np.random.seed(UNETPP_CONFIG['seed'])
torch.manual_seed(UNETPP_CONFIG['seed'])
torch.cuda.manual_seed_all(UNETPP_CONFIG['seed'])
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=False
# The released trainer does not request strict determinism. Its global pooling
# backward is unsupported under strict deterministic CUDA on some PyTorch versions.
torch.use_deterministic_algorithms(False)
unetpp_runtime['deterministic_algorithms']=torch.are_deterministic_algorithms_enabled()
unetpp_runtime['cudnn_benchmark']=torch.backends.cudnn.benchmark
unetpp_runtime['matmul_tf32']=torch.backends.cuda.matmul.allow_tf32
unetpp_runtime['cudnn_tf32']=torch.backends.cudnn.allow_tf32
unetpp_save_json('runtime.json',unetpp_runtime)
unetpp_save_json('intended_protocol.json',UNETPP_CONFIG)
print('Protocol SHA256:',hashlib.sha256((UNETPP_OUTPUT_DIR/'intended_protocol.json').read_bytes()).hexdigest())


Protocol SHA256: db8032903823cb521ebe68d41c9e922045310faefb4af0de95b579b829d90e37


## 3. Model and full-video interpolation

The following is the exact independent project module embedded for standalone
Colab execution; there is no upstream clone or MONAI installation step. Local
parity checks compare it with the pinned authors' MONAI implementation. All
source frames are used for trilinear interpolation; there is no 16-frame selection.


In [4]:
"""Independent implementation of the released ERDES UNet++ classifier protocol.

Architecture/protocol reference (read-only; no upstream source imported):
https://github.com/OSUPCVLab/ERDES/tree/4af67e0b9f4667edbae75a64a819884ee14d7a3b
The authors' encoder-only classifier uses MONAI TwoConv/Down blocks. The
equivalent operations below use PyTorch directly; this is not a full nested
segmentation UNet++. See docs/research/erdes-unetpp-protocol.md for
attribution, numerical parity checks, and paper/code discrepancies.
"""
from pathlib import Path
import time

import cv2
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F


class UNetPPRDClassifier(nn.Module):
    """Five double-convolution blocks, four pools, global pool and 256→128→1."""

    def __init__(self):
        super().__init__()
        widths = (32, 32, 64, 128, 256)
        blocks = []
        incoming = 1
        for index, outgoing in enumerate(widths):
            layers = [] if index == 0 else [nn.MaxPool3d(2, stride=2)]
            for channels_in in (incoming, outgoing):
                layers.extend([
                    nn.Conv3d(channels_in, outgoing, kernel_size=3, stride=1, padding=1, bias=True),
                    nn.InstanceNorm3d(outgoing, eps=1e-5, momentum=0.1, affine=True, track_running_stats=False),
                    nn.LeakyReLU(negative_slope=0.1, inplace=True),
                ])
            blocks.append(nn.Sequential(*layers))
            incoming = outgoing
        self.encoder = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.head = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1))

    def forward(self, inputs):
        features = self.encoder(inputs)
        return self.head(self.pool(features).flatten(1)).squeeze(1)


def resize_grayscale_volume(gray_frames, target=(96, 128, 128)):
    """Full-time-axis trilinear interpolation, not uniform frame subsampling."""
    if not isinstance(gray_frames, np.ndarray) or gray_frames.ndim != 3:
        raise ValueError('Expected a D,H,W numpy array of decoded grayscale values')
    if min(gray_frames.shape) <= 0 or len(target) != 3 or min(target) <= 0:
        raise ValueError('Source and target dimensions must be positive')
    if not np.isfinite(gray_frames).all() or gray_frames.min() < 0 or gray_frames.max() > 255:
        raise ValueError('Decoded grayscale values must be finite and in [0,255]')
    tensor = torch.from_numpy(np.ascontiguousarray(gray_frames, dtype=np.float32))[None]
    height, width = tensor.shape[-2:]
    difference = abs(height - width)
    if difference:
        before, after = difference // 2, difference - difference // 2
        padding = (0, 0, before, after) if height < width else (before, after, 0, 0)
        tensor = F.pad(tensor, padding, mode='constant', value=0)
    tensor = F.interpolate(tensor[None], size=tuple(target), mode='trilinear', align_corners=False)[0]
    tensor.div_(255.)
    minimum, maximum = tensor.min().item(), tensor.max().item()
    if not torch.isfinite(tensor).all() or minimum < -1e-6 or maximum > 1 + 1e-6:
        raise ValueError(f'Unexpected preprocessed range: {minimum}, {maximum}')
    return tensor.clamp_(0., 1.)


def prepare_unetpp_clip(path, expected_frames, expected_height=None, expected_width=None, target=(96, 128, 128)):
    """Decode all frames into one preallocated grayscale buffer, with metadata checks."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    if expected_frames < 1:
        raise ValueError('Expected frame count must be positive')
    capture = cv2.VideoCapture(str(path))
    decoded = 0
    gray = None
    try:
        if not capture.isOpened():
            raise ValueError(f'Cannot decode {path.name}')
        while True:
            ok, frame = capture.read()
            if not ok:
                break
            if frame.dtype != np.uint8 or frame.ndim != 3 or frame.shape[2] != 3:
                raise ValueError(f'{path.name}: expected uint8 three-channel decoded frame')
            height, width = frame.shape[:2]
            if ((expected_height is not None and height != expected_height)
                    or (expected_width is not None and width != expected_width)):
                raise ValueError(f'{path.name}: decoded dimensions differ from metadata')
            if decoded >= expected_frames:
                raise ValueError(f'{path.name}: decoded more frames than metadata expects')
            if gray is None:
                gray = np.empty((expected_frames, height, width), dtype=np.float32)
            if frame.shape[:2] != gray.shape[1:]:
                raise ValueError(f'{path.name}: changing frame dimensions')
            # Arithmetic channel mean is invariant to OpenCV's BGR channel order.
            gray[decoded] = frame.mean(axis=2, dtype=np.float32)
            decoded += 1
    finally:
        capture.release()
    if decoded != expected_frames:
        raise ValueError(f'{path.name}: decoded {decoded} frames; metadata expects {expected_frames}')
    return resize_grayscale_volume(gray, target)


def benchmark_cuda_batch(model_factory, inputs, labels, physical_batch, warmup=1, steps=3):
    """Disposable FP32 training/inference probe; never produces a research checkpoint."""
    if not torch.cuda.is_available():
        raise RuntimeError('A CUDA runtime is required for the hardware probe')
    if physical_batch < 1 or steps < 1 or warmup < 1:
        raise ValueError('Positive batch size, warmup and measurement steps are required')
    device = torch.device('cuda')
    model = optimizer = x = y = logits = loss = None
    initial_allocated = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats()
    try:
        model = model_factory().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1.5e-5, weight_decay=0.)
        indices = torch.arange(physical_batch) % len(inputs)
        x, y = inputs[indices].to(device), labels[indices].to(device)
        loss_function = nn.BCEWithLogitsLoss()
        train_seconds = []
        for iteration in range(warmup + steps):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.synchronize()
            start = time.perf_counter()
            logits = model(x)
            loss = loss_function(logits, y)
            if not torch.isfinite(loss):
                raise ValueError('Nonfinite hardware-probe loss')
            loss.backward()
            if iteration == 0:
                gradients = [parameter.grad for parameter in model.parameters()]
                if not all(g is not None and torch.isfinite(g).all() for g in gradients):
                    raise ValueError('Nonfinite or missing hardware-probe gradients')
                if not any(torch.count_nonzero(g).item() for g in gradients):
                    raise ValueError('All hardware-probe gradients are zero')
                del gradients
            optimizer.step()
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start
            if iteration >= warmup:
                train_seconds.append(elapsed)
        model.eval()
        inference_seconds = []
        with torch.no_grad():
            for _ in range(steps):
                torch.cuda.synchronize()
                start = time.perf_counter()
                model(x)
                torch.cuda.synchronize()
                inference_seconds.append(time.perf_counter() - start)
        return {'physical_batch': physical_batch, 'status': 'passed', 'precision': 'float32',
            'training_step_seconds_median': float(np.median(train_seconds)),
            'validation_step_seconds_median': float(np.median(inference_seconds)),
            'peak_allocated_bytes': torch.cuda.max_memory_allocated(),
            'peak_reserved_bytes': torch.cuda.max_memory_reserved(),
            'initial_allocated_bytes': initial_allocated,
            'finite_nonzero_gradients': True, 'loss_last': loss.item(),
            'weights_discarded': True}
    except torch.cuda.OutOfMemoryError:
        return {'physical_batch': physical_batch, 'status': 'out_of_memory', 'precision': 'float32',
            'weights_discarded': True}
    finally:
        del model, optimizer, x, y, logits, loss
        torch.cuda.empty_cache()


In [5]:
import hashlib
import time
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from http.client import IncompleteRead
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

def download_checked(url, path, expected_hash, algorithm='sha256', expected_bytes=None, max_attempts=5):
    path = Path(path)
    if max_attempts < 1:
        raise ValueError('max_attempts must be positive')
    if path.exists():
        content = path.read_bytes()
    else:
        for attempt in range(1, max_attempts + 1):
            retry_after = None
            try:
                with urlopen(url, timeout=90) as response:
                    content = response.read()
                break
            except HTTPError as error:
                status = error.code
                retry_after = error.headers.get('Retry-After') if error.headers else None
                error.close()
                if status not in {408, 429, 500, 502, 503, 504}:
                    raise
                failure = f'HTTP {status}'
            except (URLError, TimeoutError, ConnectionError, IncompleteRead) as error:
                failure = type(error).__name__
            if attempt == max_attempts:
                raise RuntimeError(
                    f'Download failed for {path.name}: {failure} after {max_attempts} attempts. '
                    'Rerun the acquisition cell in this runtime to reuse verified cached clips.'
                ) from None
            delay = min(60.0, float(2 ** attempt))
            if retry_after:
                try:
                    server_delay = float(retry_after)
                except ValueError:
                    try:
                        server_delay = (parsedate_to_datetime(retry_after) - datetime.now(timezone.utc)).total_seconds()
                    except (TypeError, ValueError, OverflowError):
                        server_delay = 0.0
                if server_delay > 60:
                    raise RuntimeError(
                        f'{path.name}: {failure}; server requests a wait of {server_delay:.0f}s. '
                        'Wait that long, then rerun the acquisition cell; cached clips are preserved.'
                    ) from None
                delay = max(delay, server_delay)
            print(f'{path.name}: {failure}; retry {attempt + 1}/{max_attempts} in {delay:g}s', flush=True)
            time.sleep(delay)
    if hashlib.new(algorithm, content).hexdigest() != expected_hash:
        raise ValueError(f'Checksum mismatch: {path.name}; cached file was not overwritten')
    if expected_bytes is not None and len(content) != expected_bytes:
        raise ValueError(f'Unexpected size: {path.name}')
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + '.part')
        temporary.write_bytes(content)
        temporary.replace(path)
    return content

print('Download helper ready: retry temporary HTTP failures; reuse checksum-verified cached clips.')


Download helper ready: retry temporary HTTP failures; reuse checksum-verified cached clips.


## 4. Prepare two verified official-training clips at full size

Use the first SHA256-ranked training clip within each binary class (seed 42).
Only the official training CSV is read. Cached clips are checked before reuse;
temporary download errors retry. No validation/test clips are acquired here.


In [6]:
unetpp_release='https://zenodo.org/records/18644370'
unetpp_metadata_bytes=download_checked(f'{unetpp_release}/files/erdes_metadata.csv?download=1',
    unetpp_data_dir/'erdes_metadata.csv','95828fb5bdffdbf8173b5bcfc99e8148',algorithm='md5')
unetpp_split_bytes=download_checked(f'{unetpp_release}/files/splits.zip?download=1',
    unetpp_data_dir/'splits.zip','c5630e3f056ce358dcd384aaed946e64',algorithm='md5')
unetpp_metadata=pd.read_csv(io.BytesIO(unetpp_metadata_bytes))
with zipfile.ZipFile(io.BytesIO(unetpp_split_bytes)) as unetpp_archive:
    unetpp_train_bytes=unetpp_archive.read('splits/non_rd_vs_rd/train.csv')
unetpp_train=pd.read_csv(io.BytesIO(unetpp_train_bytes))
assert len(unetpp_train)==3873 and unetpp_train['path'].is_unique
assert unetpp_train['label'].value_counts().to_dict()=={0:3511,1:362}
unetpp_training=unetpp_train.merge(unetpp_metadata,left_on='path',right_on='file_path',validate='one_to_one')
assert len(unetpp_training)==3873
assert unetpp_training['label'].map({0:'non_rd',1:'rd'}).eq(unetpp_training['diagnostic_class']).all()
unetpp_training['rank']=unetpp_training['clip_id'].map(
    lambda value:hashlib.sha256(f"{UNETPP_CONFIG['seed']}:{value}".encode()).hexdigest())
unetpp_probe_rows=unetpp_training.sort_values(['rank','clip_id']).groupby('label',sort=True).head(1).sort_values('label')
unetpp_probe_rows.to_csv(UNETPP_OUTPUT_DIR/'preflight_training_manifest.csv',index=False)
unetpp_sources={
    '164267_03674':{'sha256':'1986fa40aa2431598ffc858ef2442ae5fb16afa5a37dae047f71cc0d7687795c','bytes':264720},
    '755384_00079':{'sha256':'0b2ab744d20bef3b93a1fe99b1ae775fe3f19490d71a12904687ae9144007f7f','bytes':232445},
}
assert set(unetpp_probe_rows['clip_id'])==set(unetpp_sources)
from urllib.parse import quote
unetpp_prepared=[]
unetpp_provenance=[]
unetpp_preparation_start=time.perf_counter()
for row in unetpp_probe_rows.itertuples(index=False):
    source=unetpp_sources[row.clip_id]
    url=f"https://huggingface.co/datasets/pcvlab/erdes/resolve/{UNETPP_CONFIG['dataset_revision']}/clips/{quote(row.file_path,safe='/')}"
    local_path=unetpp_data_dir/'clips'/row.file_path
    download_checked(url,local_path,source['sha256'],expected_bytes=source['bytes'])
    tensor=prepare_unetpp_clip(local_path,int(row.frame_count),int(row.height),int(row.width))
    assert tuple(tensor.shape)==(1,96,128,128)
    unetpp_prepared.append(tensor)
    unetpp_provenance.append({'clip_id':row.clip_id,'dataset_label':int(row.label),'url':url,**source,
        'shape':list(tensor.shape),'min':tensor.min().item(),'max':tensor.max().item()})
UNETPP_INPUTS=torch.stack(unetpp_prepared)
UNETPP_LABELS=torch.tensor(unetpp_probe_rows['label'].to_numpy(),dtype=torch.float32)
unetpp_preparation_seconds=time.perf_counter()-unetpp_preparation_start
unetpp_save_json('input_checks.json',{'clips':unetpp_provenance,
    'metadata_sha256':hashlib.sha256(unetpp_metadata_bytes).hexdigest(),
    'official_train_csv_sha256':hashlib.sha256(unetpp_train_bytes).hexdigest(),
    'split_archive_sha256':hashlib.sha256(unetpp_split_bytes).hexdigest(),
    'training_manifest_sha256':hashlib.sha256((UNETPP_OUTPUT_DIR/'preflight_training_manifest.csv').read_bytes()).hexdigest(),
    'preparation_seconds_for_two_clips':unetpp_preparation_seconds,'test_used':False,
    'license_declaration':'cc-by-4.0 (author dataset card)'})
print('PASS:',tuple(UNETPP_INPUTS.shape),'range:',UNETPP_INPUTS.min().item(),UNETPP_INPUTS.max().item())


PASS: (2, 1, 96, 128, 128) range: 0.0 0.9765114784240723


## 5. Measure disposable FP32 training steps on the GPU

Measure physical batches 1, 2 and 4 with the same 96×128×128 model input. Larger
batches are skipped after an out-of-memory result. Each model is discarded;
the few optimizer updates are hardware diagnostics, not a research run. Repeated
copies of the two training clips support timing only. Do not interpret probe
losses as generalization performance. The full reproduction must initialize a
new model after its pipeline checks.


In [7]:
unetpp_count_model=UNetPPRDClassifier()
unetpp_parameter_count=sum(parameter.numel() for parameter in unetpp_count_model.parameters())
assert unetpp_parameter_count==UNETPP_CONFIG['model']['expected_parameter_count']
del unetpp_count_model
unetpp_profiles=[]
for physical_batch in UNETPP_CONFIG['preflight']['physical_batches']:
    gc.collect()
    print('Measuring physical batch',physical_batch,flush=True)
    result=benchmark_cuda_batch(UNetPPRDClassifier,UNETPP_INPUTS,UNETPP_LABELS,physical_batch,
        warmup=UNETPP_CONFIG['preflight']['warmup_steps'],steps=UNETPP_CONFIG['preflight']['measured_steps'])
    unetpp_profiles.append(result)
    unetpp_save_json('hardware_profiles.json',unetpp_profiles)
    print(result,flush=True)
    if result['status']=='out_of_memory':
        print('Larger batches deferred; smaller completed profiles remain available.')
        break
unetpp_successful=[profile for profile in unetpp_profiles if profile['status']=='passed']
if not unetpp_successful:
    raise RuntimeError('No FP32 physical batch passed. Keep hardware_profiles.json; do not launch full training.')
unetpp_fastest=min(unetpp_successful,key=lambda value:value['training_step_seconds_median']/value['physical_batch'])
unetpp_batch=unetpp_fastest['physical_batch']
unetpp_effective=UNETPP_CONFIG['intended_training']['reference_effective_batch']
assert unetpp_effective%unetpp_batch==0
unetpp_accumulation=unetpp_effective//unetpp_batch
unetpp_train_compute=3873/unetpp_batch*unetpp_fastest['training_step_seconds_median']
unetpp_val_compute=431/unetpp_batch*unetpp_fastest['validation_step_seconds_median']
unetpp_cache_bytes=(3873+431)*96*128*128*4
import shutil
unetpp_disk_free=shutil.disk_usage(unetpp_data_dir).free
unetpp_summary={'stage':'hardware_preflight_only','runtime':unetpp_runtime,
    'parameter_count':unetpp_parameter_count,'profiles':unetpp_profiles,
    'candidate_physical_batch':unetpp_batch,'candidate_accumulation_steps':unetpp_accumulation,
    'intended_effective_batch':unetpp_effective,
    'rough_training_compute_seconds_per_epoch':unetpp_train_compute,
    'rough_validation_compute_seconds_per_epoch':unetpp_val_compute,
    'rough_50_epoch_compute_hours':50*(unetpp_train_compute+unetpp_val_compute)/3600,
    'timing_caveat':'Microbenchmark extrapolation only; excludes full decoding/cache reads/checkpoint IO/disconnects; update frequency differs under accumulation.',
    'projected_float32_train_validation_cache_bytes':unetpp_cache_bytes,
    'current_runtime_disk_free_bytes':unetpp_disk_free,
    'prepared_cache_fits_current_disk':unetpp_cache_bytes<unetpp_disk_free,
    'full_training_performed':False,'benchmark_weights_discarded':True,'test_used':False}
unetpp_save_json('preflight_summary.json',unetpp_summary)
display(pd.DataFrame(unetpp_profiles))
print('Candidate physical batch:',unetpp_batch,'| accumulation steps:',unetpp_accumulation)
print('Rough 50-epoch COMPUTE projection:',round(unetpp_summary['rough_50_epoch_compute_hours'],2),'hours')
print('This is not a guaranteed wall-clock duration. Full-data IO and checkpoint resume must be planned.')
print('Full train/validation float32 cache:',round(unetpp_cache_bytes/1e9,2),'GB; free runtime disk:',round(unetpp_disk_free/1e9,2),'GB')
print('Saved:',UNETPP_OUTPUT_DIR/'preflight_summary.json')


Measuring physical batch 1
{'physical_batch': 1, 'status': 'passed', 'precision': 'float32', 'training_step_seconds_median': 0.434257826999783, 'validation_step_seconds_median': 0.08273303600003601, 'peak_allocated_bytes': 1693088256, 'peak_reserved_bytes': 2027945984, 'initial_allocated_bytes': 0, 'finite_nonzero_gradients': True, 'loss_last': 0.6725238561630249, 'weights_discarded': True}
Measuring physical batch 2
{'physical_batch': 2, 'status': 'passed', 'precision': 'float32', 'training_step_seconds_median': 0.6683877240000129, 'validation_step_seconds_median': 0.165191170000071, 'peak_allocated_bytes': 3310451200, 'peak_reserved_bytes': 3978297344, 'initial_allocated_bytes': 18087936, 'finite_nonzero_gradients': True, 'loss_last': 0.6781684160232544, 'weights_discarded': True}
Measuring physical batch 4
{'physical_batch': 4, 'status': 'passed', 'precision': 'float32', 'training_step_seconds_median': 1.2213691730000846, 'validation_step_seconds_median': 0.3299774519998664, 'peak_a

,physical_batch,status,precision,training_step_seconds_median,validation_step_seconds_median,peak_allocated_bytes,peak_reserved_bytes,initial_allocated_bytes,finite_nonzero_gradients,loss_last,weights_discarded
0,1,passed,float32,0.434258,0.082733,1693088256,2027945984,0,True,0.672524,True
1,2,passed,float32,0.668388,0.165191,3310451200,3978297344,18087936,True,0.678168,True
2,4,passed,float32,1.221369,0.329977,6543866368,7952400384,18087936,True,0.675937,True


Candidate physical batch: 4 | accumulation steps: 12
Rough 50-epoch COMPUTE projection: 16.92 hours
This is not a guaranteed wall-clock duration. Full-data IO and checkpoint resume must be planned.
Full train/validation float32 cache: 27.08 GB; free runtime disk: 120.8 GB
Saved: /content/drive/MyDrive/rd-ultrasound-ml/outputs/unetpp_resource_check/20261101T193236373065Z/preflight_summary.json


## 6. Preserve the report and choose the full-data run

Share `preflight_summary.json` and the printed candidate batch/timing report.
The next training configuration must use the measured microbatch, correct
accumulation-window normalization, all training/validation clips, 50 epochs,
and resumable checkpoints. A wall-clock estimate needs full-epoch IO timing,
not just this short compute probe. Do not launch a speculative multi-hour job
or substitute the 16-frame pilot's inputs for the paper input.

When Drive is enabled, reports are already in the Drive folder printed in
section 1; the ZIP below is also saved beside that run folder in Drive. Browse
Google Drive in your browser to download it. No clinical or unseen-patient
claim, test result, or completed reproduction is produced by this notebook.


In [8]:
unetpp_zip=UNETPP_OUTPUT_DIR.parent/(UNETPP_OUTPUT_DIR.name+'_preflight.zip')
with zipfile.ZipFile(unetpp_zip,'w',zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(UNETPP_OUTPUT_DIR.iterdir()):
        if file.is_file() and file.suffix in {'.json','.csv'}:
            archive.write(file,arcname=f'{UNETPP_OUTPUT_DIR.name}/{file.name}')
print('Preflight ZIP saved:',unetpp_zip)
if str(unetpp_zip).startswith('/content/drive/MyDrive/'):
    print('Google Drive location: My Drive/'+str(unetpp_zip).removeprefix('/content/drive/MyDrive/'))
display(FileLink(str(unetpp_zip)))


Preflight ZIP saved: /content/drive/MyDrive/rd-ultrasound-ml/outputs/unetpp_resource_check/20261101T193236373065Z_preflight.zip
Google Drive location: My Drive/rd-ultrasound-ml/outputs/unetpp_resource_check/20261101T193236373065Z_preflight.zip


/content/drive/MyDrive/rd-ultrasound-ml/outputs/unetpp_resource_check/20261101T193236373065Z_preflight.zip